# Observables: Diagnosing Phases in Symmetry-Resolved Exact Diagonalization

**Abstract.** Symmetry-resolved exact diagonalization gives eigenvalues and eigenvectors labelled by symmetry quantum numbers. The next question is: *what do we measure to identify the phase?* This notebook is a pedagogical, hands-on tour of the observable toolkit in `realspace_exactdiagonalization_py`, run throughout on the **bosonic Haldane honeycomb FCI** ($t''=-0.58$) on the $[2,3]$ torus with $n=3$ hard-core bosons (fast; the two nearly-degenerate semion ground states live in the momentum sectors $k=(0,0)$ and $k=(1,0)$). We cover, in order:

1. **Density distribution** — `vertices_occupation_distribution_full_ed` (open boundary conditions only), and *why* $\langle n_i\rangle$ is mathematically useless for translation-invariant PBC systems;
2. **Connected static structure factor** $S(q)$ — `static_structure_factor`, `structure_factor_allowed_momenta`, `compute_structure_factor_map`, `plot_structure_factor_map` (charge-order diagnostic);
3. **Off-diagonal long-range order** (ODLRO) $\rho(k)$ — `off_diagonal_long_range_order`, `compute_odlro_map` (superfluidity / condensation diagnostic);
4. **Entanglement spectra** — `entanglement_spectrum` and `particle_entanglement_spectrum` (Schmidt decomposition, the semion-doublet degenerate pairs);
5. **Many-body Chern number** — `many_body_chern_number` (non-Abelian Fukui–Hatsugai–Suzuki flux-torus formula, with an honest caveat about its numerical fragility at small flux grids).

Every number printed below is **cross-checked against the Julia reference** `RealSpace_ExactDiagonalization.jl` to 12 significant digits.

**References.**

1. XDiag: *Exact Diagonalization for Quantum Many-Body Systems*, [arXiv:2505.02901](https://arxiv.org/abs/2505.02901).
2. T. Fukui, Y. Hatsugai, H. Suzuki, *Chern Numbers in Discretized Brillouin Zone*, J. Phys. Soc. Jpn. **74**, 1674 (2005).
3. R. Resta, *Quantum-Mechanical Position Operator in Extended Systems*, Phys. Rev. Lett. **80**, 1800 (1998).
4. R. B. Laughlin, *Quantized Hall conductivity in two dimensions*, Phys. Rev. B **23**, 5632 (1981).
5. D. N. Sheng, Z.-C. Gu, K. Sun, L. Sheng, *Fractional quantum Hall effect in the absence of Landau levels*, Phys. Rev. Lett. **107**, 146803 (2011).


## Motivation and Scope

For translation-invariant Hamiltonians with periodic boundary conditions, many naive diagnostics fail for *fundamental, not numerical* reasons. This notebook explains why, and provides the correct tools to distinguish:

- **Fractional Chern insulators** (FCI),
- **Charge-density waves / Wigner crystals** (CDW),
- **Superfluids** (SF), and
- **Supersolids** (SS).

The benchmark system is the hard-core boson model on the Haldane honeycomb lattice at half filling of the lower Chern band ($\nu=1/2$ per band). On the $2\times3$ torus (12 flattened vertices, 3 bosons) the exact spectrum shows **two nearly-degenerate ground states** — the finite-size fingerprint of the bosonic Laughlin/semion state (topological ground-state degeneracy GSD$=2$ on the torus, [Sheng *et al.*, PRL **107**, 146803 (2011)]).


## Why $\langle n_i\rangle$ is Useless for Translation-Invariant PBC Systems

### The fundamental problem

Consider a translation-invariant Hamiltonian on a finite lattice with periodic boundary conditions:

\begin{equation}
[H, T_{\bm R}] = 0, \qquad \forall \text{ lattice translations } T_{\bm R}.
\end{equation}

A **non-degenerate** ground state $|\psi\rangle$ of any such Hamiltonian is necessarily a simultaneous eigenstate of all translations:

\begin{equation}
T_{\bm R} |\psi\rangle = e^{i\bm k\cdot\bm R} |\psi\rangle.
\end{equation}

Consequently, the one-point density is **identically uniform**:

\begin{equation}
\boxed{\;\langle n_i\rangle = \langle\psi|T_{-\bm R}\,n_i\,T_{\bm R}|\psi\rangle = \langle n_{i+\bm R}\rangle \;\Rightarrow\; \langle n_i\rangle \equiv \frac{N_e}{N}.\;}
\end{equation}

This is **not** a sign that the system is in a fluid phase. It is a mathematical identity holding for **any** translation-invariant eigenstate — CDW, FCI, superfluid, everything. The CDW order is hidden by the quantum superposition of all translation-related patterns, so a one-point observable in a *single* momentum sector cannot see it.

### Consequence for the API

The single-sector density operator is guaranteed uniform for translation-invariant PBC systems, so the package **removes** the single-sector density API and keeps only `vertices_occupation_distribution_full_ed`, which **asserts open boundary conditions** (`not any(pbc_indicator)`). Only on a disc/open chain — where translational symmetry is genuinely broken by the boundaries — can a single ground-state eigenvector display real-space density modulation.


In [ ]:
import math
import os
from fractions import Fraction

import numpy as np

import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__))))

# Bosonic Haldane honeycomb FCI, t'' = -0.58, [2,3] torus, 3 hard-core bosons.
sample_size = [2, 3]
model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_DNSheng)
lattice = model.lattice
n_filled = 3   # half filling of the Chern band -> 1/4 of the 12 vertices

G = ed.build_translation_group(lattice)
ed_data = ed.build_ed_data(
    model, filling_fraction=Fraction(n_filled, lattice.n_site), symmetry_group=G)

print(f"Full Hilbert space dim: {math.comb(lattice.n_site, n_filled)}")
print(f"Orbits: {len(ed_data.orbit_catalog.representative_mask_list)}")
print(f"Momenta: {[ir.label for ir in ed_data.irrep_list]}")

ed.ed_scan(ed_data, nev=5, mode="matrix")   # ~1 min first run (Numba JIT)
print("E0(k=(0,0)) =", float(ed_data.ed_scan_res[0][0][0]))


## (a) Density Distribution — `vertices_occupation_distribution_full_ed`

The full real-space ED occupation $\langle n_i\rangle$ is computed from the *identity*-group eigenvector (each symmetry orbit = one raw Fock mask). Because the identity group has no translation projector, a *single* eigenvector is not forced to be a momentum eigenstate — but for a PBC translation-invariant Hamiltonian the ground state still is one (when non-degenerate), so the function guards against PBC misuse.

On an **open chain**, however, the boundary genuinely breaks translation symmetry and $\langle n_i\rangle$ shows real-space modulation. The cell below (i) demonstrates the PBC guard and (ii) computes $\langle n_i\rangle$ for an open 1D chain of hard-core bosons to show the difference.


In [ ]:
from tightbinding_py import (
    add_hopping_term,
    generate_bilinear_terms,
    initialize_real_space_lattice,
    initialize_real_space_tightbinding_model,
)
from realspace_exactdiagonalization_py import (
    Particle_Statistics,
    Real_Space_Second_Quantized_Model,
)

# Guard first: the PBC Haldane model must REFUSE a one-point density.
try:
    ed.vertices_occupation_distribution_full_ed(
        model, filling_fraction=Fraction(n_filled, lattice.n_site))
except AssertionError as err:
    print("PBC guard (expected):", err)

# Now an open 1D chain: 12 sites, nearest-neighbor hopping -t, 3 hard-core bosons.
L = 12
lat = initialize_real_space_lattice(
    sample_size=[L, 1],
    brav_vec_list=[[1.0, 0.0], [0.0, 1.0]],
    sub_crys_list=[[0.0, 0.0]],
    lattice_name="1D_open_chain",
    pbc_indicator=[False, False],   # OBC -> genuine real-space modulation
)
tb = initialize_real_space_tightbinding_model(lat, model_name="chain_obc")
add_hopping_term(tb, ((((0, 0), 1), ((1, 0), 1)), -1.0))
obc_model = Real_Space_Second_Quantized_Model(
    params={"t": 1.0},
    lattice=lat,
    tb_model=tb,
    particle_statistics=Particle_Statistics.BOSONIC,
    bilinear_terms=generate_bilinear_terms(tb, twisted_phases_over_2π=[0.0, 0.0]),
    density_density_terms=[],
)

density = ed.vertices_occupation_distribution_full_ed(
    obc_model, filling_fraction=Fraction(3, L), target_eigval_idx=1)
print("⟨n_i⟩ on the open chain (1-based):")
print(np.round(density, 6))
print(f"spread max-min = {float(density.max() - density.min()):.6f}  "
      "(nonzero -> genuine edge modulation, impossible with PBC)")


## (b) Connected Static Structure Factor $S(q)$ — Charge Order

The **connected** density–density correlation function in momentum space:

\begin{equation}
\boxed{\;S^{\alpha\beta}(q) = \frac{1}{N}\sum_{i,j} e^{i q\cdot(r_i-r_j)}
\big(\langle n_i^\alpha n_j^\beta\rangle - \langle n_i^\alpha\rangle\langle n_j^\beta\rangle\big).\;}
\end{equation}

$\alpha,\beta$ are optional flavour labels (e.g. sublattice A vs B); the default selects the total density. The **connected** correlator $\langle n_i n_j\rangle_c = \langle n_i n_j\rangle - \langle n_i\rangle\langle n_j\rangle$ carries genuine two-point charge correlations *even within a single translation sector* — unlike $\langle n_i\rangle$. A CDW/Wigner crystal shows sharp Bragg peaks at the ordering wavevector $Q$; an FCI or superfluid shows a smooth, featureless $S(q)$. The result below is smooth (no Bragg peak) — consistent with the FCI ground state at $k=(0,0)$.


In [ ]:
# S(q) on the finite-torus allowed momenta (uniform reciprocal grid).
q_points, S_q = ed.static_structure_factor(
    model, (0, 0), filling_fraction=Fraction(1, 4), ed_data=ed_data)

print("Allowed momenta and S(q):")
for q, s in zip(q_points, S_q):
    print(f"  q=({q[0]:.6f}, {q[1]:.6f})   S(q)={s:.12f}")
print(f"sum_q S(q) = {float(np.sum(S_q)):.12f}")

# Same data, folded into the first BZ, as (qx, qy, S_q):
qx, qy, S_allowed = ed.structure_factor_allowed_momenta(
    model, (0, 0), filling_fraction=Fraction(1, 4), ed_data=ed_data)
print("structure_factor_allowed_momenta ->", len(qx), "points")


**Verified against Julia** (bosonic Haldane $[2,3]$, $n=3$, $k=(0,0)$): $S(q{=}(0,0))=0$, $S(q{=}(3.1416,-1.8138))=0.235827519433$, and $\sum_q S(q) = 0.855372576870$ — agreement to 12 digits.


In [ ]:
# Dense S(q) heatmap over [-1.5π, 1.5π]² with the first-BZ boundary overlaid.
kx, ky, S_map = ed.compute_structure_factor_map(
    model, (0, 0), filling_fraction=Fraction(1, 4), k_resolution=61, ed_data=ed_data)
print("S_map shape:", S_map.shape)

fig, ax = ed.plot_structure_factor_map(
    model, (0, 0), filling_fraction=Fraction(1, 4), k_resolution=61,
    ed_data=ed_data)


## (c) Off-Diagonal Long-Range Order $\rho(k)$ — Superfluidity

The one-body density matrix and its Fourier transform (the **momentum distribution**):

\begin{equation}
\boxed{\;\rho_{ij} = \langle a_i^\dagger a_j\rangle,\;}
\qquad
\boxed{\;\rho(k) = \frac{1}{N}\sum_{i,j} e^{ik\cdot(r_i-r_j)}\,\langle a_i^\dagger a_j\rangle.\;}
\end{equation}

- The eigenvalues of $\rho_{ij}$ are the **natural-orbital occupations**. A superfluid has one (or a few) macroscopic eigenvalues $\sim O(N_e)$ — the Penrose–Onsager criterion for Bose condensation.
- $\rho(k)$ shows **where** condensation occurs: a sharp peak at $k^*$ with weight $\sim O(N_e)$.
- For an FCI or CDW insulator, $\rho(k)$ is broad with all values $\sim O(1)$.

For hard-core bosons $\rho_{ij}$ is a pure real-space object: $a_i^\dagger a_j$ hops a particle from $j$ to $i$. In a translation sector a fixed local operator is not symmetry-invariant, so the implementation loops over every ket-orbit member, applies the one-body hop, canonicalizes the scattered mask, and reads the bra coefficient from `project_to_sector` — algebraically identical to a full-Fock expansion but without materializing the amplitude dictionary.

**Important:** $\rho(k)$ and the Fourier transform of $\langle n_i\rangle$ are completely different objects. $\rho(k)$ measures off-diagonal (phase) coherence; $\mathcal F[\langle n_i\rangle]$ measures diagonal (density) modulation. Only $\rho(k)$ detects superfluidity.


In [ ]:
odlro = ed.off_diagonal_long_range_order(
    model, (0, 0), filling_fraction=Fraction(1, 4), ed_data=ed_data)

# 0-based indices: odlro[0,0] = ρ[1,1], odlro[0,1] = ρ[1,2] in Julia's 1-based G.
print("ρ[0,0] = ⟨n_1⟩      =", float(np.real(odlro[0, 0])))
print("ρ[0,1] = ⟨a_1† a_2⟩ =", float(np.real(odlro[0, 1])))
print("tr ρ   = Σ_i ⟨n_i⟩  =", float(np.real(np.trace(odlro))))

kx, ky, rho_map = ed.compute_odlro_map(
    model, (0, 0), filling_fraction=Fraction(1, 4), k_resolution=61, ed_data=ed_data)

fig, ax = ed.plot_odlro_map(
    model, (0, 0), filling_fraction=Fraction(1, 4), k_resolution=61,
    ed_data=ed_data)


**Verified against Julia**: $G[1,1] = 0.25$ (uniform $\langle n_1\rangle = N_e/N = 3/12$) and $G[1,2] = 0.242411509340$ — agreement to 12 digits. The diagonal exactly reproduces the uniform density, while the off-diagonal element is the short-range one-body coherence.


## (d) Entanglement Spectrum — the Semion Doublet

### Schmidt decomposition

Bipartition the sites into $A$ (here sites $1\ldots6$) and its complement $B$. Write the ground state in Fock bases $|a\rangle_A$, $|b\rangle_B$ and arrange the amplitudes in a matrix $M_{ab}$:

\begin{equation}
|\psi\rangle = \sum_{a,b} M_{ab}\,|a\rangle_A |b\rangle_B .
\end{equation}

The singular-value decomposition $M = U\,\mathrm{diag}(\lambda^{1/2})\,V^\dagger$ gives the **Schmidt form**

\begin{equation}
|\psi\rangle = \sum_\alpha \lambda_\alpha^{1/2}\, |\alpha\rangle_A |\beta\rangle_B ,
\qquad \sum_\alpha \lambda_\alpha = 1 ,
\end{equation}

with **entanglement energies** $\xi_\alpha = -\log \lambda_\alpha$. The block structure at fixed particle number $N_A$ in region $A$ is resolved automatically. For the semion FCI, the reduced density matrix of a single torus ground state shows a **doubly degenerate** low-lying entanglement spectrum — the fingerprint of the nontrivial topological order (each Schmidt level comes in the semion doublet).

### Momentum-resolved particle entanglement spectrum (PES)

`particle_entanglement_spectrum` instead traces out $N_B = N - N_A$ *particles* and block-diagonalizes the reduced density matrix in the many-body momentum sectors of subsystem $A$. The level counting below the entanglement gap encodes the quasihole counting of the Laughlin state.


In [ ]:
# Spatial entanglement spectrum, partition A = flattened sites 1..6.
res = ed.entanglement_spectrum(
    model, (0, 0), partition_a=list(range(1, 7)),
    filling_fraction=Fraction(1, 4), ed_data=ed_data)

print("Entanglement energies ξ = -log λ (top 12, sorted):")
for i, xi in enumerate(res.entanglement_energies[:12], start=1):
    print(f"  ξ{i:2d} = {xi:.12f}")
print(f"norm_probability = {res.norm_probability:.15f}")

fig, ax = ed.plot_entanglement_spectrum(res)


**Verified against Julia** (partition A = sites 1–6): the ten lowest entanglement energies are the five degenerate semion pairs
$$0.717377793288,\ 0.717377793288,\ 5.275522802330,\ 5.275522802330,\ 5.822579678276,\ 5.822579678276,\ 5.834253167054,\ 5.834253167054,\ 6.950569544066,\ 6.950569544066,$$
with `norm_probability ≈ 1.0` — agreement to 12 digits.


In [ ]:
# Momentum-resolved PES: trace out 2 particles, keep N_A = 1 in subsystem A.
pes = ed.particle_entanglement_spectrum(
    model, [(0, 0), (1, 0)], n_particles_a=1,
    filling_fraction=Fraction(1, 4), ed_data=ed_data)

print("PES (N_A=1): top levels sorted by ξ = -log λ:")
for row in pes.levels[:10]:
    print(f"  k={row.momentum}  level={row.level}  ξ={row.entanglement_energy:.12f}  "
          f"p={row.probability:.12f}")
print(f"norm_probability = {pes.norm_probability:.15f}")

fig, ax = ed.plot_particle_entanglement_spectrum(pes)


## (e) Many-Body Chern Number — Non-Abelian Flux-Torus Formula

On the torus, thread fluxes $\theta = (\theta_x,\theta_y)$ through the two periodic cycles and follow a selected low-energy **manifold** of states $\{|\psi_m(\theta)\rangle\}$. The non-Abelian (Fukui–Hatsugai–Suzuki) U(1) link between neighboring flux points is the phase of the overlap-matrix determinant:

\begin{equation}
U_\mu(\theta) = \frac{\det S(\theta, \theta+\delta\mu)}{|\det S(\theta, \theta+\delta\mu)|},
\qquad S_{mn}(\theta,\theta') = \langle\psi_m(\theta)|\psi_n(\theta')\rangle ,
\end{equation}

and the plaquette Berry curvature is

\begin{equation}
F(\theta) = \Im\ln\!\Big[ U_x(\theta)\, U_y(\theta+\delta x)\, U_x(\theta+\delta y)^{-1}\, U_y(\theta)^{-1}\Big],
\qquad
C = \frac{1}{2\pi}\sum_{\theta} F(\theta) .
\end{equation}

For the two-state FCI manifold $\{k=(0,0),\,(1,0)\}$ this should give the Laughlin pair's $C=1$ in the thermodynamic limit.

### ⚠️ Honest fragility caveat

At the small $7\times7$ flux grid on the $[2,3]$ torus, the **non-Abelian Chern number of the nearly-degenerate semion manifold is numerically fragile**: the level-1 tracking of the two quasi-degenerate states across flux points can swap members, and the link determinant $\det S$ (with `min_link_det ≈ 0.087`) is not comfortably close to 1. Both languages agree *exactly* here — Julia gives $C = 7.1\times10^{-17}$ and Python gives $C = 0.000000$ — but the textbook $C=1$ for the Laughlin pair requires a larger/cleaner setup (larger systems, or explicit manifold states with robust level tracking). We present the formula and the *raw* number honestly rather than a rounded artifact.


In [ ]:
import contextlib
import io

CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")

# 7×7 flux grid over the two-state FCI manifold [(0,0), (1,0)].
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    chern = ed.many_body_chern_number(
        model, [(0, 0), (1, 0)],
        filling_fraction=Fraction(1, 4),
        flux_grid_size=(7, 7),
        nev_per_sector=1,
        mode="matrix",
        checkpoint_dir=CKPT_DIR,
    )

print("flux grid:", chern.flux_grid_size)
print(f"C            = {chern.chern_number:.6f}")
print(f"round(C)     = {chern.rounded_chern}")
print(f"min |det S|  = {chern.min_link_det:.6f}")


## Summary of Verified Numbers (Python vs Julia, 12 digits)

| Observable | Julia / Python (this notebook) |
|---|---|
| $S(q{=}(0,0))$ | $0$ |
| $S(q{=}(3.1416,-1.8138))$ | $0.235827519433$ |
| $\sum_q S(q)$ | $0.855372576870$ |
| ODLRO $G[1,1]$ | $0.25$ |
| ODLRO $G[1,2]$ | $0.242411509340$ |
| Entanglement energies (10 lowest) | five degenerate semion pairs (see above) |
| `norm_probability` | $\approx 1.0$ |
| Many-body Chern $C$ ($7\times7$ flux) | $0.000000$ (Julia $7.1\times10^{-17}$), `min_link_det ≈ 0.087` |

---

*This notebook is part of `realspace_exactdiagonalization_py`. The observable implementations live in `src/realspace_exactdiagonalization_py/observables/`; the model builder in `src/realspace_exactdiagonalization_py/models/bosonic_fci.py`.*
